# 03 - Comandos DML em Tabelas Delta Lake

Demonstra operações DML (**INSERT**, **UPDATE**, **DELETE**) em tabelas Delta Lake armazenadas no MinIO, além de recursos como **HISTORY** e **TIME TRAVEL**.

**Pré-requisitos:** Notebook `02` executado (tabelas Delta no bucket `bronze`).

## 1. Configuração e SparkSession

In [1]:
import os
from dotenv import load_dotenv
from pyspark.sql import SparkSession
from delta import *
from delta.tables import DeltaTable

load_dotenv(override=True)

MINIO_ENDPOINT   = os.getenv('MINIO_ENDPOINT')
MINIO_ACCESS_KEY = os.getenv('MINIO_ACCESS_KEY')
MINIO_SECRET_KEY = os.getenv('MINIO_SECRET_KEY')
BRONZE_BUCKET    = os.getenv('MINIO_BRONZE_BUCKET')

spark = (
    SparkSession.builder
    .appName('DML_Delta_Lake')
    .master('local[*]')
    .config('spark.jars.packages', 'io.delta:delta-spark_2.12:3.2.0,org.apache.hadoop:hadoop-aws:3.3.4')
    .config('spark.sql.extensions', 'io.delta.sql.DeltaSparkSessionExtension')
    .config('spark.sql.catalog.spark_catalog', 'org.apache.spark.sql.delta.catalog.DeltaCatalog')
    .config('spark.hadoop.fs.s3a.endpoint', MINIO_ENDPOINT)
    .config('spark.hadoop.fs.s3a.access.key', MINIO_ACCESS_KEY)
    .config('spark.hadoop.fs.s3a.secret.key', MINIO_SECRET_KEY)
    .config('spark.hadoop.fs.s3a.path.style.access', 'true')
    .config('spark.hadoop.fs.s3a.impl', 'org.apache.hadoop.fs.s3a.S3AFileSystem')
    .config('spark.hadoop.fs.s3a.connection.ssl.enabled', 'false')
    .getOrCreate()
)
print('SparkSession criada com sucesso!')
spark

:: loading settings :: url = jar:file:/usr/local/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-243eb63e-7ffd-4811-be6f-1062235a83d0;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.0 in central
	found io.delta#delta-storage;3.2.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
:: resolution report :: resolve 242ms :: artifacts dl 7ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.262 from central in [default]
	io.delta#delta-spark_2.12;3.2.0 from central in [default]
	io.delta#delta-storage;3.2.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	org.apache.hadoop#hadoop-aws;3

SparkSession criada com sucesso!


## 2. Registrar Tabelas Delta como SQL Tables

In [2]:
# Registrar as tabelas Delta Lake para uso com Spark SQL
tabelas_delta = ['anexo', 'cidade', 'estado', 'ouvidoria', 'servico_afetado', 'tipo_ouvidoria', 'usuario']

for tabela in tabelas_delta:
    delta_path = f's3a://{BRONZE_BUCKET}/{tabela}'
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {tabela}
        USING delta
        LOCATION '{delta_path}'
    """)

# Listar tabelas registradas
print('Tabelas registradas no Spark:')
spark.sql('SHOW TABLES').show(truncate=False)

26/05/01 23:09:27 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


Tabelas registradas no Spark:
+---------+---------------+-----------+
|namespace|tableName      |isTemporary|
+---------+---------------+-----------+
|default  |anexo          |false      |
|default  |cidade         |false      |
|default  |estado         |false      |
|default  |ouvidoria      |false      |
|default  |servico_afetado|false      |
|default  |tipo_ouvidoria |false      |
|default  |usuario        |false      |
+---------+---------------+-----------+



## 3. Consultar Dados Atuais (SELECT)

In [3]:
# Visualizar as tabelas de domínio
print('=== CIDADES ===')
spark.sql('SELECT * FROM cidade ORDER BY id_cidade').show()

print('=== OUVIDORIAS ===')
spark.sql('SELECT * FROM ouvidoria ORDER BY id_ouvidoria').show()

print('=== TIPOS DE OUVIDORIA (primeiros 10) ===')
spark.sql('SELECT * FROM tipo_ouvidoria ORDER BY id_tipo LIMIT 10').show()

=== CIDADES ===


26/05/01 23:09:31 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+---------+--------------------+----------+
|id_cidade|         nome_cidade|cod_estado|
+---------+--------------------+----------+
|        1|      Afonso Cláudio|         8|
|        2|  Água Doce do Norte|         8|
|        3|        Águia Branca|         8|
|        4|              Alegre|         8|
|        5|      Alfredo Chaves|         8|
|        6|       Alto Rio Novo|         8|
|        7|            Anchieta|         8|
|        8|              Apiacá|         8|
|        9|             Aracruz|         8|
|       10|     Atilio Vivacqua|         8|
|       11|        Baixo Guandu|         8|
|       12|Barra de São Fran...|         8|
|       13|       Boa Esperança|         8|
|       14|  Bom Jesus do Norte|         8|
|       15|           Brejetuba|         8|
|       16|Cachoeiro de Itap...|         8|
|       17|           Cariacica|         8|
|       18|             Castelo|         8|
|       19|            Colatina|         8|
|       20|  Conceição da Barra|

In [4]:
# Contagem de registros por tabela
print(f'{"Tabela":<15} {"Registros":>10}')
print('-' * 27)
for tabela in tabelas_delta:
    count = spark.sql(f'SELECT COUNT(*) as cnt FROM {tabela}').collect()[0]['cnt']
    print(f'{tabela:<15} {count:>10}')

Tabela           Registros
---------------------------


26/05/01 23:09:35 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


anexo                    2
cidade                 500
estado                  27
ouvidoria                3
servico_afetado          8
tipo_ouvidoria           5
usuario                  3


---
## 4. INSERT - Inserir Novos Registros

Vamos inserir novos registros nas tabelas `marca`, `modelo` e `cliente`.

In [5]:
# INSERT - Novos serviços afetados (para as novas marcas)
print('--- INSERT: Novos serviços afetados ---')

spark.sql("""
    INSERT INTO servico_afetado (id_servico, nome_servico) 
    VALUES 
        (6, 'Segurança Pública e Defesa Civil'),
        (7, 'Obras e Infraestrutura'),
        (8, 'Meio Ambiente e Sustentabilidade'),
        (9, 'Cultura, Esporte e Lazer'),
        (10, 'Transporte e Trânsito')
""")

spark.sql("SELECT * FROM servico_afetado WHERE id_servico >= 6 ORDER BY id_servico").show()
print('5 novos modelos inseridos!')

--- INSERT: Novos serviços afetados ---
+----------+--------------------+
|id_servico|        nome_servico|
+----------+--------------------+
|         6|Segurança Pública...|
|         6|Assistência Socia...|
|         7|Obras e Infraestr...|
|         7|Planejamento e De...|
|         8|Meio Ambiente e S...|
|         8|              Outros|
|         9|Cultura, Esporte ...|
|        10|Transporte e Trân...|
+----------+--------------------+

5 novos modelos inseridos!


In [6]:
# INSERT - Novas ouvidorias (para os novos serviços afetados)
print('--- INSERT: Novas Ouvidorias ---')
spark.sql("SELECT COUNT(*) as antes FROM ouvidoria").show()

spark.sql("""
    INSERT INTO ouvidoria (id_ouvidoria, descricao_ouvidoria, cod_tipo, cod_servico, protocolo_ouvidoria, data_ouvidoria, cod_usuario) 
    VALUES 
        (4, 'Solicitação de maior frequência de rondas policiais no bairro durante o período noturno.', 3, 6, '202605030001', '2026-05-03 08:00:00', 1),
        (5, 'Reclamação sobre o atraso nas obras de recapeamento da via principal.', 4, 7, '202605030002', '2026-05-03 09:15:00', 2),
        (6, 'Denúncia de despejo ilegal de resíduos tóxicos no rio da região.', 5, 8, '202605030003', '2026-05-03 10:30:00', 3),
        (7, 'Elogio à organização do festival cultural ocorrido no último fim de semana.', 2, 9, '202605030004', '2026-05-03 11:45:00', 1),
        (8, 'Sugestão para a instalação de novos semáforos no cruzamento da zona sul.', 1, 10, '202605030005', '2026-05-03 13:00:00', 2);
""")

spark.sql("SELECT * FROM ouvidoria ORDER BY id_ouvidoria").show()
print('5 novas ouvidorias inseridas!')

--- INSERT: Novas Ouvidorias ---
+-----+
|antes|
+-----+
|    3|
+-----+

+------------+--------------------+--------+-----------+-------------------+-------------------+-----------+
|id_ouvidoria| descricao_ouvidoria|cod_tipo|cod_servico|protocolo_ouvidoria|     data_ouvidoria|cod_usuario|
+------------+--------------------+--------+-----------+-------------------+-------------------+-----------+
|           1|Falta de medicame...|       4|          2|       202605010001|2026-05-01 08:30:00|          1|
|           2|Ótimo atendimento...|       2|          1|       202605010002|2026-05-01 09:45:00|          2|
|           3|Solicito o reparo...|       3|          7|       202605010003|2026-05-01 14:20:00|          3|
|           4|Solicitação de ma...|       3|          6|       202605030001|2026-05-03 08:00:00|          1|
|           5|Reclamação sobre ...|       4|          7|       202605030002|2026-05-03 09:15:00|          2|
|           6|Denúncia de despe...|       5|          

---
## 5. UPDATE - Atualizar Registros

Vamos atualizar registros existentes nas tabelas.

In [7]:
# UPDATE - Atualizar descrição de ouvidoria
print('--- UPDATE: Ajustar descrição de ouvidoria ---')
print('ANTES:')
spark.sql("SELECT * FROM ouvidoria WHERE id_ouvidoria = 1").show()

spark.sql("""
    UPDATE ouvidoria SET descricao_ouvidoria = 'Falta de medicamentos nos postos de saúde.' WHERE id_ouvidoria = 1
""")

print('DEPOIS:')
spark.sql("SELECT * FROM ouvidoria WHERE id_ouvidoria = 1").show()

--- UPDATE: Ajustar descrição de ouvidoria ---
ANTES:
+------------+--------------------+--------+-----------+-------------------+-------------------+-----------+
|id_ouvidoria| descricao_ouvidoria|cod_tipo|cod_servico|protocolo_ouvidoria|     data_ouvidoria|cod_usuario|
+------------+--------------------+--------+-----------+-------------------+-------------------+-----------+
|           1|Falta de medicame...|       4|          2|       202605010001|2026-05-01 08:30:00|          1|
+------------+--------------------+--------+-----------+-------------------+-------------------+-----------+

DEPOIS:
+------------+--------------------+--------+-----------+-------------------+-------------------+-----------+
|id_ouvidoria| descricao_ouvidoria|cod_tipo|cod_servico|protocolo_ouvidoria|     data_ouvidoria|cod_usuario|
+------------+--------------------+--------+-----------+-------------------+-------------------+-----------+
|           1|Falta de medicame...|       4|          2|       20

In [8]:
# UPDATE - Atualizar nome de serviço
print('--- UPDATE: Atualizar serviço ---')
print('ANTES:')
spark.sql("SELECT * FROM servico_afetado WHERE id_servico = 5").show()

spark.sql("""
    UPDATE servico_afetado
    SET nome_servico = 'Agropecuária'
    WHERE id_servico = 5
""")

print('DEPOIS:')
spark.sql("SELECT * FROM servico_afetado WHERE id_servico = 5").show()

--- UPDATE: Atualizar serviço ---
ANTES:
+----------+------------+
|id_servico|nome_servico|
+----------+------------+
|         5| Agricultura|
+----------+------------+

DEPOIS:
+----------+------------+
|id_servico|nome_servico|
+----------+------------+
|         5|Agropecuária|
+----------+------------+



In [9]:
# UPDATE com DeltaTable API (alternativa ao SQL)
print('--- UPDATE via DeltaTable API ---')
from pyspark.sql.functions import lit

dt_marca = DeltaTable.forPath(spark, f's3a://{BRONZE_BUCKET}/servico_afetado')

dt_marca.update(
    condition="id_servico = 1",
    set={"nome_servico": lit("Educação, Ciência, Tecnologia, etc.")}
)

spark.sql("SELECT * FROM servico_afetado WHERE id_servico = 1").show()

--- UPDATE via DeltaTable API ---
+----------+--------------------+
|id_servico|        nome_servico|
+----------+--------------------+
|         1|Educação, Ciência...|
+----------+--------------------+



---
## 6. DELETE - Remover Registros

Vamos deletar registros de tabelas Delta.

In [10]:
# DELETE - Remover ouvidorias sem ID via SQL
print('--- DELETE: Remover ouvidorias sem ID ---')
print('ANTES:')
spark.sql("SELECT * FROM ouvidoria ORDER BY id_ouvidoria").show()

spark.sql("DELETE FROM ouvidoria WHERE id_ouvidoria is null")

print('DEPOIS:')
spark.sql("SELECT * FROM ouvidoria ORDER BY id_ouvidoria").show()

--- DELETE: Remover ouvidorias sem ID ---
ANTES:
+------------+--------------------+--------+-----------+-------------------+-------------------+-----------+
|id_ouvidoria| descricao_ouvidoria|cod_tipo|cod_servico|protocolo_ouvidoria|     data_ouvidoria|cod_usuario|
+------------+--------------------+--------+-----------+-------------------+-------------------+-----------+
|           1|Falta de medicame...|       4|          2|       202605010001|2026-05-01 08:30:00|          1|
|           2|Ótimo atendimento...|       2|          1|       202605010002|2026-05-01 09:45:00|          2|
|           3|Solicito o reparo...|       3|          7|       202605010003|2026-05-01 14:20:00|          3|
|           4|Solicitação de ma...|       3|          6|       202605030001|2026-05-03 08:00:00|          1|
|           5|Reclamação sobre ...|       4|          7|       202605030002|2026-05-03 09:15:00|          2|
|           6|Denúncia de despe...|       5|          8|       202605030003|202

In [11]:
# DELETE - Remover ouvidoria via SQL
print('--- DELETE: Remover cliente teste ---')
spark.sql("SELECT * FROM ouvidoria WHERE id_ouvidoria >= 6 order by id_ouvidoria").show()

spark.sql("DELETE FROM ouvidoria WHERE id_ouvidoria = 8")

print('Apos DELETE:')
spark.sql("SELECT * FROM ouvidoria WHERE id_ouvidoria >= 6 order by id_ouvidoria").show()

--- DELETE: Remover cliente teste ---
+------------+--------------------+--------+-----------+-------------------+-------------------+-----------+
|id_ouvidoria| descricao_ouvidoria|cod_tipo|cod_servico|protocolo_ouvidoria|     data_ouvidoria|cod_usuario|
+------------+--------------------+--------+-----------+-------------------+-------------------+-----------+
|           6|Denúncia de despe...|       5|          8|       202605030003|2026-05-03 10:30:00|          3|
|           7|Elogio à organiza...|       2|          9|       202605030004|2026-05-03 11:45:00|          1|
|           8|Sugestão para a i...|       1|         10|       202605030005|2026-05-03 13:00:00|          2|
+------------+--------------------+--------+-----------+-------------------+-------------------+-----------+

Apos DELETE:
+------------+--------------------+--------+-----------+-------------------+-------------------+-----------+
|id_ouvidoria| descricao_ouvidoria|cod_tipo|cod_servico|protocolo_ouvidoria|

In [12]:
# DELETE via DeltaTable API
print('--- DELETE via DeltaTable API ---')
dt_ouvidoria = DeltaTable.forPath(spark, f's3a://{BRONZE_BUCKET}/ouvidoria')

print('Ouvidorias >= 5 ANTES:')
spark.sql("SELECT * FROM ouvidoria WHERE id_ouvidoria >= 5").show()

dt_ouvidoria.delete("id_ouvidoria = 6")

print('Ouvidorias >= 5 DEPOIS:')
spark.sql("SELECT * FROM ouvidoria WHERE id_ouvidoria >= 5").show()

--- DELETE via DeltaTable API ---
Ouvidorias >= 5 ANTES:
+------------+--------------------+--------+-----------+-------------------+-------------------+-----------+
|id_ouvidoria| descricao_ouvidoria|cod_tipo|cod_servico|protocolo_ouvidoria|     data_ouvidoria|cod_usuario|
+------------+--------------------+--------+-----------+-------------------+-------------------+-----------+
|           7|Elogio à organiza...|       2|          9|       202605030004|2026-05-03 11:45:00|          1|
|           5|Reclamação sobre ...|       4|          7|       202605030002|2026-05-03 09:15:00|          2|
|           6|Denúncia de despe...|       5|          8|       202605030003|2026-05-03 10:30:00|          3|
+------------+--------------------+--------+-----------+-------------------+-------------------+-----------+

Ouvidorias >= 5 DEPOIS:
+------------+--------------------+--------+-----------+-------------------+-------------------+-----------+
|id_ouvidoria| descricao_ouvidoria|cod_tipo|co

---
## 7. HISTORY - Histórico de Versões Delta

O Delta Lake mantém um log de transações que permite visualizar o histórico completo de alterações.

In [13]:
# Histórico da tabela marca
print('=== HISTORICO DA TABELA OUVIDORIA ===')
dt_ouvidoria = DeltaTable.forPath(spark, f's3a://{BRONZE_BUCKET}/ouvidoria')
dt_ouvidoria.history().select('version', 'timestamp', 'operation', 'operationMetrics').show(truncate=False)

=== HISTORICO DA TABELA OUVIDORIA ===
+-------+-------------------+---------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|version|timestamp          |operation|operationMetrics                                                                                                                                                                                                                                                                                                            |
+-------+-------------------+---------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [14]:
# Histórico via Spark SQL
print('=== HISTORICO DA TABELA servico_afetado ===')
spark.sql('DESCRIBE HISTORY servico_afetado').select('version', 'timestamp', 'operation').show(truncate=False)

=== HISTORICO DA TABELA servico_afetado ===
+-------+-------------------+---------+
|version|timestamp          |operation|
+-------+-------------------+---------+
|3      |2026-05-01 23:09:44|UPDATE   |
|2      |2026-05-01 23:09:43|UPDATE   |
|1      |2026-05-01 23:09:38|WRITE    |
|0      |2026-05-01 23:09:07|WRITE    |
+-------+-------------------+---------+



---
## 8. TIME TRAVEL - Viagem no Tempo

O Delta Lake permite ler versões anteriores dos dados.

In [15]:
# Versão atual da tabela ouvidoria
print('=== OUVIDORIA - VERSAO ATUAL ===')
spark.sql('SELECT * FROM ouvidoria ORDER BY id_ouvidoria').show()

=== OUVIDORIA - VERSAO ATUAL ===
+------------+--------------------+--------+-----------+-------------------+-------------------+-----------+
|id_ouvidoria| descricao_ouvidoria|cod_tipo|cod_servico|protocolo_ouvidoria|     data_ouvidoria|cod_usuario|
+------------+--------------------+--------+-----------+-------------------+-------------------+-----------+
|           1|Falta de medicame...|       4|          2|       202605010001|2026-05-01 08:30:00|          1|
|           2|Ótimo atendimento...|       2|          1|       202605010002|2026-05-01 09:45:00|          2|
|           3|Solicito o reparo...|       3|          7|       202605010003|2026-05-01 14:20:00|          3|
|           4|Solicitação de ma...|       3|          6|       202605030001|2026-05-03 08:00:00|          1|
|           5|Reclamação sobre ...|       4|          7|       202605030002|2026-05-03 09:15:00|          2|
|           7|Elogio à organiza...|       2|          9|       202605030004|2026-05-03 11:45:00

In [16]:
# Time Travel - Ler versão 0 (estado original antes de qualquer DML)
print('=== OUVIDORIA - VERSAO 0 (original) ===')
df_v0 = spark.read.format('delta').option('versionAsOf', 0).load(f's3a://{BRONZE_BUCKET}/ouvidoria')
df_v0.orderBy('id_ouvidoria').show()

=== OUVIDORIA - VERSAO 0 (original) ===
+------------+--------------------+--------+-----------+-------------------+-------------------+-----------+
|id_ouvidoria| descricao_ouvidoria|cod_tipo|cod_servico|protocolo_ouvidoria|     data_ouvidoria|cod_usuario|
+------------+--------------------+--------+-----------+-------------------+-------------------+-----------+
|           1|Falta de medicame...|       4|          2|       202605010001|2026-05-01 08:30:00|          1|
|           2|Ótimo atendimento...|       2|          1|       202605010002|2026-05-01 09:45:00|          2|
|           3|Solicito o reparo...|       3|          7|       202605010003|2026-05-01 14:20:00|          3|
+------------+--------------------+--------+-----------+-------------------+-------------------+-----------+



In [17]:
# Time Travel - Comparar versão original vs atual
print('=== COMPARACAO: Versao 0 vs Atual ===')
df_original = spark.read.format('delta').option('versionAsOf', 0).load(f's3a://{BRONZE_BUCKET}/ouvidoria')
df_atual = spark.read.format('delta').load(f's3a://{BRONZE_BUCKET}/ouvidoria')

print(f'Versao 0: {df_original.count()} registros')
print(f'Atual:    {df_atual.count()} registros')

# Mostrar registros que foram adicionados (existem no atual, mas não na v0)
print('\nRegistros ADICIONADOS:')
df_atual.subtract(df_original).show()

=== COMPARACAO: Versao 0 vs Atual ===
Versao 0: 3 registros
Atual:    6 registros

Registros ADICIONADOS:
+------------+--------------------+--------+-----------+-------------------+-------------------+-----------+
|id_ouvidoria| descricao_ouvidoria|cod_tipo|cod_servico|protocolo_ouvidoria|     data_ouvidoria|cod_usuario|
+------------+--------------------+--------+-----------+-------------------+-------------------+-----------+
|           4|Solicitação de ma...|       3|          6|       202605030001|2026-05-03 08:00:00|          1|
|           7|Elogio à organiza...|       2|          9|       202605030004|2026-05-03 11:45:00|          1|
|           5|Reclamação sobre ...|       4|          7|       202605030002|2026-05-03 09:15:00|          2|
|           1|Falta de medicame...|       4|          2|       202605010001|2026-05-01 08:30:00|          1|
+------------+--------------------+--------+-----------+-------------------+-------------------+-----------+



# ---
# 9. Resumo Final
# ---

In [ ]:
print('=' * 70)
print('RESUMO DAS OPERAÇÕES DML REALIZADAS NO DATA LAKE (OUVIDORIA)')
print('=' * 70)
print()
print('INSERT:')
print('  - 5 novas ouvidorias inseridas (IDs 4, 5, 6, 7 e 8)')
print()
print('UPDATE:')
print('  - ouvidoria ID 1 -> Descrição atualizada para "Falta de medicamentos..." (via SQL)')
print('  - servico_afetado ID 5 -> Nome atualizado para "Agropecuária" (via SQL)')
print('  - servico_afetado ID 1 -> Nome atualizado para "Educação, Ciência..." (via DeltaTable API)')
print()
print('DELETE:')
print('  - ouvidorias com ID nulo (sujeira) removidas (via SQL)')
print('  - ouvidoria ID 8 (sugestão de semáforos) removida (via SQL)')
print('  - ouvidoria ID 6 (denúncia de resíduos) removida (via DeltaTable API)')
print()
print('HISTORY e TIME TRAVEL:')
print('  - Histórico completo de transações consultado (DESCRIBE HISTORY)')
print('  - Leitura do estado original dos dados (versionAsOf = 0)')
print('  - Comparação e extração de diferenças entre Versão 0 vs Versão Atual')
print('=' * 70)

RESUMO DAS OPERACOES DML REALIZADAS

INSERT:
  - 3 novas marcas (TESLA, BYD, GWM)
  - 5 novos modelos (MODEL 3, MODEL Y, DOLPHIN, etc.)
  - 2 novos clientes

UPDATE:
  - marca TESLA -> TESLA MOTORS (via SQL)
  - marca BYD -> BYD AUTO (via DeltaTable API)
  - cliente 99001 nome e CPF atualizados

DELETE:
  - marca GWM removida (via SQL)
  - cliente 99002 removido (via SQL)
  - modelo HAVAL H6 removido (via DeltaTable API)

HISTORY e TIME TRAVEL:
  - Historico completo de transacoes
  - Leitura de versoes anteriores (versionAsOf)
  - Comparacao entre versoes


In [19]:
spark.stop()
print('SparkSession finalizada.')

SparkSession finalizada.
